# Silver - Vendedores

Padronização de cadastros de vendedores e associação de canais.

In [ ]:
%run ../0_Config/0-Init

In [ ]:
# Parâmetros de Inicialização
sistema = 'case'
table_name = 'vendedores'
input_path = f"{var_bronze}/{sistema}/{table_name}/data"
output_path_data = f"{var_silver}/{sistema}/{table_name}/data"
table_name_schema = f'{var_environment}.{var_silver_schema}.{sistema}_{table_name}'

In [ ]:
from pyspark.sql.functions import col, trim, initcap, lower

df_bronze = spark.read.format("delta").load(input_path)

df_clean = (
    df_bronze
    .select(
        col("id_vendedor").cast("integer").alias("id_vendedor"),
        initcap(trim(col("nome"))).cast("string").alias("nome_vendedor"),
        col("canal_id").cast("integer").alias("id_canal"),
        lower(trim(col("email"))).cast("string").alias("email_vendedor")
    )
    .filter(col("id_vendedor").isNotNull())
    .dropDuplicates(["id_vendedor"])
)

In [ ]:
process_data(
    df_write=df_clean,
    tipo_carga='full',
    nome_gravacao_tabela=table_name_schema,
    caminho_gravacao_tabela=output_path_data,
    chave_clusterby=['id_canal'],
    chave_upsert='id_vendedor'
)